In [2]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
import pandas as pd

BASE_URL = "https://api.met.gov.my/v2.1"
TOKEN = "800b8992bc918dff3b95348ea036e3cf2837b92c"

headers = {
    "Authorization": f"METToken {TOKEN}"
}

location_category = "TOWN"
offset = 0

url = f"{BASE_URL}/locations"
params = {
    "locationcategoryid": location_category,
    "offset": offset
}

response = requests.get(url, headers=headers, params=params)

print("Status Code:", response.status_code)

data = response.json()
data

Status Code: 200


{'metadata': {'resultset': {'count': 154,
   'offset': 0,
   'limit': 50,
   'locationcategoryid': 'TOWN'}},
 'results': [{'id': 'LOCATION:127',
   'name': 'MUAR',
   'locationcategoryid': 'TOWN',
   'locationrootid': 'LOCATION:1',
   'latitude': 2.0442,
   'longitude': 102.5689},
  {'id': 'LOCATION:126',
   'name': 'TANGKAK',
   'locationcategoryid': 'TOWN',
   'locationrootid': 'LOCATION:1',
   'latitude': 2.2673,
   'longitude': 102.5453},
  {'id': 'LOCATION:125',
   'name': 'LABIS',
   'locationcategoryid': 'TOWN',
   'locationrootid': 'LOCATION:1',
   'latitude': 2.385,
   'longitude': 103.021},
  {'id': 'LOCATION:124',
   'name': 'JOHOR BAHRU',
   'locationcategoryid': 'TOWN',
   'locationrootid': 'LOCATION:1',
   'latitude': 1.4655,
   'longitude': 103.7578},
  {'id': 'LOCATION:122',
   'name': 'AYER HITAM',
   'locationcategoryid': 'TOWN',
   'locationrootid': 'LOCATION:1',
   'latitude': 1.915,
   'longitude': 103.1808},
  {'id': 'LOCATION:123',
   'name': 'BATU PAHAT',
   'lo

In [4]:
results = data.get("results", [])
df_locations = pd.json_normalize(results)

df_locations

,id,name,locationcategoryid,locationrootid,latitude,longitude
0,LOCATION:127,MUAR,TOWN,LOCATION:1,2.044200,102.568900
1,LOCATION:126,TANGKAK,TOWN,LOCATION:1,2.267300,102.545300
2,LOCATION:125,LABIS,TOWN,LOCATION:1,2.385000,103.021000
3,LOCATION:124,JOHOR BAHRU,TOWN,LOCATION:1,1.465500,103.757800
4,LOCATION:122,AYER HITAM,TOWN,LOCATION:1,1.915000,103.180800
5,LOCATION:123,BATU PAHAT,TOWN,LOCATION:1,1.854800,102.932500
6,LOCATION:131,MERSING,TOWN,LOCATION:1,2.431200,103.840500
7,LOCATION:659,ISKANDAR PUTERI,TOWN,LOCATION:1,1.413590,103.631719
8,LOCATION:129,KLUANG,TOWN,LOCATION:1,2.025100,103.332800
9,LOCATION:768,NUSAJAYA,TOWN,LOCATION:1,1.423900,103.648500


In [5]:
df_locations.columns.tolist()

['id', 'name', 'locationcategoryid', 'locationrootid', 'latitude', 'longitude']

In [6]:
useful_cols = [col for col in [
    "locationid",
    "locationname",
    "locationcategoryid",
    "latitude",
    "longitude",
    "locationrootid",
    "locationrootname"
] if col in df_locations.columns]

df_locations[useful_cols]

,locationcategoryid,latitude,longitude,locationrootid
0,TOWN,2.044200,102.568900,LOCATION:1
1,TOWN,2.267300,102.545300,LOCATION:1
2,TOWN,2.385000,103.021000,LOCATION:1
3,TOWN,1.465500,103.757800,LOCATION:1
4,TOWN,1.915000,103.180800,LOCATION:1
5,TOWN,1.854800,102.932500,LOCATION:1
6,TOWN,2.431200,103.840500,LOCATION:1
7,TOWN,1.413590,103.631719,LOCATION:1
8,TOWN,2.025100,103.332800,LOCATION:1
9,TOWN,1.423900,103.648500,LOCATION:1


In [7]:
def get_all_locations(location_category: str, token: str) -> pd.DataFrame:
    base_url = "https://api.met.gov.my/v2.1/locations"
    headers = {
        "Authorization": f"METToken {token}"
    }

    all_results = []
    offset = 0
    page_size = 50

    while True:
        params = {
            "locationcategoryid": location_category,
            "offset": offset
        }

        response = requests.get(base_url, headers=headers, params=params)
        response.raise_for_status()

        data = response.json()
        results = data.get("results", [])

        if not results:
            break

        all_results.extend(results)

        print(f"Fetched {len(results)} records at offset={offset}")

        if len(results) < page_size:
            break

        offset += page_size

    return pd.json_normalize(all_results)

In [ ]:
df_town_locations = get_all_locations("TOWN", TOKEN)

# df_town_locations.head()
df_town_locations


Fetched 50 records at offset=0
Fetched 50 records at offset=50
